# 01 - PubMedQA Dataset Exploration

This notebook loads the PubMedQA labeled dataset and explores its structure,
distributions, and characteristics before downstream processing.

**Outputs:** Dataset statistics, label distributions, context length analysis, MeSH term frequencies.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from collections import Counter
from src.data_loader import load_pubmedqa, get_dataset_stats, build_mesh_lookup, get_meshes_with_frequency

## Load Dataset

In [ ]:
df = load_pubmedqa()
print(f"Dataset shape: {df.shape}")
df.head()

## Dataset Statistics

In [ ]:
stats = get_dataset_stats(df)

print(f"Total samples: {stats['total_samples']}")
print(f"Mean context length: {stats['context_length_mean']:.3f}")
print(f"\nLabel distribution: {stats['label_distribution']}")
print(f"\nContext length distribution: {stats['context_length_distribution']}")
print(f"\nUnique MeSH terms: {stats['unique_meshes']}")

## Context Structure Inspection

In [ ]:
# Inspect context structure for a single sample
sample = df.loc[0, "context"]
print("Keys:", list(sample.keys()))
print(f"\nNumber of contexts: {len(sample['contexts'])}")
print(f"Labels: {list(sample['labels'])}")
print(f"MeSH terms: {list(sample['meshes'])}")
print(f"\nFirst context (truncated): {sample['contexts'][0][:200]}...")

## Context Length Distribution

In [ ]:
context_lengths = df["context"].apply(lambda x: len(x["contexts"]))
print(f"Context lengths - Mean: {context_lengths.mean():.2f}, Median: {context_lengths.median()}, Min: {context_lengths.min()}, Max: {context_lengths.max()}")
print(f"\nValue counts:\n{context_lengths.value_counts().sort_index()}")

## Top MeSH Terms

In [ ]:
print("Top 40 MeSH terms:")
for term, count in stats['top_meshes']:
    print(f"  {term}: {count}")

In [ ]:
# Mid-frequency MeSH terms (potential specialized domains)
mid_freq = get_meshes_with_frequency(df, min_freq=5, max_freq=20)
print(f"MeSH terms with frequency 5-20: {len(mid_freq)} terms")
list(mid_freq.items())[:20]

## Build MeSH Lookup

This mapping is used later for metadata enrichment during chunking.

In [ ]:
mesh_lookup = build_mesh_lookup(df)
print(f"MeSH lookup entries: {len(mesh_lookup)}")
print(f"Sample: pubid={list(mesh_lookup.keys())[0]} -> {list(mesh_lookup.values())[0][:5]}...")

## Save Processed DataFrame

Save the loaded dataset for downstream notebooks to avoid re-downloading.

In [ ]:
df.to_pickle("../data/processed/pubmedqa_train.pkl")
print("Saved to data/processed/pubmedqa_train.pkl")